In [ ]:
import requests
import json
import time

# 您的 API Token
api_token = '949751a4ea6a586f9e2805a3909d456a-c-app'

# 请求头
headers = {
    'Content-Type': 'application/json'
}

# 黄金的代码
symbol = 'XAUUSD'  # 请根据 AllTick 的产品代码确认黄金的代码

# 每次请求的 K 线数量
batch_size = 1000

# 获取当前时间的时间戳（秒）
end_timestamp = int(time.time())

# 存储所有 K 线数据
all_data = []

# 计算需要的请求次数
total_k_lines = 10080  # 一周的 K 线数量
num_requests = (total_k_lines + batch_size - 1) // batch_size

for _ in range(num_requests):
    # 请求参数
    params = {
        "trace": "python_http_test1",
        "data": {
            "code": "XAUUSD",
            "kline_type": 1,  # 1 表示 1 分钟 K 线
            "kline_timestamp_end": end_timestamp,
            "query_kline_num": batch_size,
            "adjust_type": 0  # 复权类型，0 表示不复权
        }
    }

    # 将参数转换为 JSON 格式并进行 URL 编码
    query = json.dumps(params)
    query = requests.utils.quote(query)

    # 构建请求 URL
    url = f'https://quote.alltick.io/quote-b-api/kline?token={api_token}&query={query}'

    try:
        # 发送请求
        response = requests.get(url, headers=headers)
        # 检查响应状态码
        if response.status_code == 200:
            data = response.json()
            if 'data' in data and 'kline' in data['data']:
                kline_data = data['data']['kline']
                all_data.extend(kline_data)
                # 更新结束时间戳为最早的 K 线时间戳，以获取更早的数据
                if kline_data:
                    end_timestamp = kline_data[-1][0] // 1000  # 转换为秒
                else:
                    break
            else:
                print("响应中缺少关键数据。")
                break
        else:
            print(f"请求失败，状态码：{response.status_code}")
            break
    except Exception as e:
        print(f"请求出现异常：{e}")
        break

    # 避免频率限制，暂停一段时间
    time.sleep(1)

# 处理获取的所有 K 线数据
print(f"共获取到 {len(all_data)} 条 K 线数据")
